# PPO Pipeline B (15-trial walk-forward random search)

15-trial random search per window across six walk-forward windows. Replays the saved per-window trial logs (pipeline_b_w{1..6}_ppo_trials.csv) to pick the best validation config per window, then loads the corresponding final retrained agent from models_walkforward/ to recompute the test-year metrics. Writes the PPO rows of pipeline_b_trials.csv, pipeline_b_test_results.csv, pipeline_b_summary.csv, and pipeline_b_summary.xlsx (read-merge with the DQN rows from Notebook 6).


# Imports


In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
from datetime import datetime

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from scipy import stats

warnings.filterwarnings('ignore')


# Configuration


In [ ]:
EPISODE_LENGTH = 252
TRANSACTION_COST = 0.001
REWARD_SCALE = 100

N_TRIALS = 15
TIMESTEPS_CHOICES = [250_000, 500_000, 1_000_000]
LR_LOG_LOW = -5    # log10(1e-5)
LR_LOG_HIGH = -2   # log10(1e-2)

FEATURE_COLS = ["PRC", "VOL", "RET", "SHROUT", "OPENPRC", "vix_index",
                "RSI", "MACD", "BB_PCT", "ATR", "rolling_vol_60",
                "SMA_20", "SMA_50", "log_vol"]

TRAIN_DATA_PATH = 'files/train_data_v3.csv.gz'
VAL_DATA_PATH = 'files/val_data_v3.csv.gz'
TEST_DATA_PATH = 'files/test_data_v3.csv.gz'
SCALER_PARAMS_PATH = 'results/scaler_params.csv'

MODEL_DIR = 'models_walkforward'
TRIALS_CSV = 'results/pipeline_b_trials.csv'
TEST_RESULTS_CSV = 'results/pipeline_b_test_results.csv'
SUMMARY_CSV = 'results/pipeline_b_summary.csv'
SUMMARY_XLSX = 'results/pipeline_b_summary.xlsx'

WINDOWS = [
    {'id': 1, 'train_start': '1999-01-01', 'train_end': '2013-12-31',
            'val_start':   '2014-01-01', 'val_end':   '2017-12-31',
            'test_start':  '2018-01-01', 'test_end':  '2018-12-31'},
    {'id': 2, 'train_start': '2000-01-01', 'train_end': '2014-12-31',
            'val_start':   '2015-01-01', 'val_end':   '2018-12-31',
            'test_start':  '2019-01-01', 'test_end':  '2019-12-31'},
    {'id': 3, 'train_start': '2001-01-01', 'train_end': '2015-12-31',
            'val_start':   '2016-01-01', 'val_end':   '2019-12-31',
            'test_start':  '2020-01-01', 'test_end':  '2020-12-31'},
    {'id': 4, 'train_start': '2002-01-01', 'train_end': '2016-12-31',
            'val_start':   '2017-01-01', 'val_end':   '2020-12-31',
            'test_start':  '2021-01-01', 'test_end':  '2021-12-31'},
    {'id': 5, 'train_start': '2003-01-01', 'train_end': '2017-12-31',
            'val_start':   '2018-01-01', 'val_end':   '2021-12-31',
            'test_start':  '2022-01-01', 'test_end':  '2022-12-31'},
    {'id': 6, 'train_start': '2004-01-01', 'train_end': '2018-12-31',
            'val_start':   '2019-01-01', 'val_end':   '2022-12-31',
            'test_start':  '2023-01-01', 'test_end':  '2023-12-31'},
]


# Environment


In [ ]:
class PortfolioEnv(gym.Env):
    """Continuous-action portfolio environment for PPO."""
    
    metadata = {"render_modes": []}
    TRANSACTION_COST = TRANSACTION_COST
    
    def __init__(self, df, episode_length=252, mode='train'):
        super().__init__()
        self.mode = mode
        self.episode_length = episode_length
        
        self.dates = sorted(df['date'].unique())
        self.stocks = sorted(df['PERMNO'].unique())
        self.n_assets = len(self.stocks)
        
        self.feature_cols = ["PRC","VOL","RET","SHROUT","OPENPRC","vix_index",
                             "RSI","MACD","BB_PCT","ATR","rolling_vol_60",
                             "SMA_20","SMA_50","log_vol"]
        self.n_features = len(self.feature_cols)
        
        self.obs_matrix = np.zeros((len(self.dates), self.n_assets, self.n_features))
        
        if 'raw_RET' in df.columns:
            ret_col = 'raw_RET'
        else:
            ret_col = 'RET'
            print("  WARNING: Using normalized RET for rewards")
        
        self.ret_matrix = np.zeros((len(self.dates), self.n_assets))
        
        date_to_idx = {d: i for i, d in enumerate(self.dates)}
        stock_to_idx = {s: i for i, s in enumerate(self.stocks)}
        
        for _, row in df.iterrows():
            d_idx = date_to_idx[row['date']]
            s_idx = stock_to_idx[row['PERMNO']]
            self.obs_matrix[d_idx, s_idx] = row[self.feature_cols].values.astype(np.float64)
            self.ret_matrix[d_idx, s_idx] = row[ret_col]
        
        # FIX: Clean NaN/Inf from observation matrix
        nan_count = np.isnan(self.obs_matrix).sum()
        inf_count = np.isinf(self.obs_matrix).sum()
        if nan_count > 0 or inf_count > 0:
            print(f"  WARNING: Found {nan_count} NaN and {inf_count} Inf values in obs_matrix — replacing with 0")
            self.obs_matrix = np.nan_to_num(self.obs_matrix, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Clip extreme values to prevent gradient explosion
        obs_clip = 10.0
        clipped = np.sum(np.abs(self.obs_matrix) > obs_clip)
        if clipped > 0:
            pct = clipped / self.obs_matrix.size * 100
            print(f"  Clipping {clipped} values ({pct:.2f}%) outside [-{obs_clip}, {obs_clip}]")
            self.obs_matrix = np.clip(self.obs_matrix, -obs_clip, obs_clip)
        
        # Also clean return matrix
        nan_ret = np.isnan(self.ret_matrix).sum()
        if nan_ret > 0:
            print(f"  WARNING: Found {nan_ret} NaN values in ret_matrix — replacing with 0")
            self.ret_matrix = np.nan_to_num(self.ret_matrix, nan=0.0)
        
        # Verify no remaining issues
        assert not np.isnan(self.obs_matrix).any(), "obs_matrix still has NaN!"
        assert not np.isinf(self.obs_matrix).any(), "obs_matrix still has Inf!"
        
        obs_dim = self.n_assets * self.n_features + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.Box(-1, 1, shape=(self.n_assets,), dtype=np.float32)
        
        self.weights = np.ones(self.n_assets) / self.n_assets
        self.current_step = 0
        self.start_step = 0
    
    def _get_obs(self):
        features = self.obs_matrix[self.current_step].flatten()
        obs = np.concatenate([features, self.weights]).astype(np.float32)
        # Final safety check
        obs = np.nan_to_num(obs, nan=0.0, posinf=0.0, neginf=0.0)
        return obs
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if self.mode == 'train':
            max_start = len(self.dates) - self.episode_length - 1
            self.start_step = self.np_random.integers(0, max(1, max_start))
        else:
            self.start_step = 0
        self.current_step = self.start_step
        self.weights = np.ones(self.n_assets) / self.n_assets
        return self._get_obs(), {}
    
    def step(self, action):
        # Softmax to get valid portfolio weights
        action = np.nan_to_num(action, nan=0.0)  # safety
        exp_a = np.exp(action - np.max(action))
        weights = exp_a / (exp_a.sum() + 1e-8)  # epsilon to prevent div by zero
        
        turnover = np.sum(np.abs(weights - self.weights))
        tc = self.TRANSACTION_COST * turnover
        
        self.current_step += 1
        returns = self.ret_matrix[self.current_step]
        portfolio_return = np.dot(weights, returns) - tc
        reward = portfolio_return * REWARD_SCALE
        
        # Clip reward to prevent extreme gradients
        reward = np.clip(reward, -10, 10)
        
        self.weights = weights
        steps_taken = self.current_step - self.start_step
        done = (steps_taken >= self.episode_length) or \
               (self.current_step >= len(self.dates) - 1)
        
        info = {'portfolio_return': portfolio_return, 'turnover': turnover}
        return self._get_obs(), float(reward), done, False, info


# Evaluation


In [ ]:
def compute_metrics(daily_returns):
    """Compute standard portfolio metrics from a daily return series."""
    daily_returns = np.asarray(daily_returns)
    n_days = len(daily_returns)
    if n_days == 0:
        return {'cumulative_return': 0, 'ann_return': 0, 'ann_volatility': 0,
                'sharpe': 0, 'sortino': 0, 'max_drawdown': 0, 'calmar': 0,
                'win_rate': 0, 'n_trading_days': 0}

    cumulative = np.prod(1 + daily_returns) - 1
    ann_return = (1 + cumulative) ** (252 / max(n_days, 1)) - 1
    ann_vol = np.std(daily_returns) * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0

    downside = daily_returns[daily_returns < 0]
    downside_vol = np.std(downside) * np.sqrt(252) if len(downside) > 0 else 0.001
    sortino = ann_return / downside_vol if downside_vol > 0 else 0

    cum_curve = np.cumprod(1 + daily_returns)
    running_max = np.maximum.accumulate(cum_curve)
    drawdowns = (cum_curve - running_max) / running_max
    max_dd = float(np.min(drawdowns))

    calmar = ann_return / abs(max_dd) if abs(max_dd) > 0 else 0
    win_rate = float(np.mean(daily_returns > 0))

    return {
        'cumulative_return': round(cumulative * 100, 2),
        'ann_return': round(ann_return * 100, 2),
        'ann_volatility': round(ann_vol * 100, 2),
        'sharpe': round(float(sharpe), 4),
        'sortino': round(float(sortino), 4),
        'max_drawdown': round(max_dd * 100, 2),
        'calmar': round(float(calmar), 4),
        'win_rate': round(win_rate * 100, 2),
        'n_trading_days': n_days,
    }


def evaluate_model(model, eval_df):
    """Run a trained model deterministically over eval_df and return metrics + daily returns."""
    n_days = len(eval_df['date'].unique())
    env = PortfolioEnv(eval_df, episode_length=n_days - 2, mode='eval')

    obs, _ = env.reset()
    daily_returns = []
    turnovers = []

    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        daily_returns.append(info['portfolio_return'])
        turnovers.append(info['turnover'])
        done = done or truncated

    metrics = compute_metrics(daily_returns)
    metrics['mean_daily_turnover'] = round(float(np.mean(turnovers)), 4) if turnovers else 0
    return metrics, np.array(daily_returns)


def run_baselines_on_period(df):
    """Compute Equal_Weight, Buy_Hold and MktCap_Weight baselines on a date range."""
    dates = sorted(df['date'].unique())
    stocks = sorted(df['PERMNO'].unique())
    n_assets = len(stocks)
    stock_to_idx = {s: i for i, s in enumerate(stocks)}

    ret_col = 'raw_RET' if 'raw_RET' in df.columns else 'RET'
    ret_matrix = np.zeros((len(dates), n_assets))
    for _, row in df.iterrows():
        d_idx = dates.index(row['date'])
        s_idx = stock_to_idx[row['PERMNO']]
        ret_matrix[d_idx, s_idx] = row[ret_col]

    baselines = {}

    # Equal-Weight (rebalanced daily)
    w = np.ones(n_assets) / n_assets
    rets = [np.dot(w, ret_matrix[t]) for t in range(1, len(dates))]
    baselines['Equal_Weight'] = np.array(rets)

    # Buy-and-Hold (initial equal weight, then drift)
    w = np.ones(n_assets) / n_assets
    rets = []
    for t in range(1, len(dates)):
        rets.append(np.dot(w, ret_matrix[t]))
        w = w * (1 + ret_matrix[t])
        if w.sum() > 0:
            w /= w.sum()
    baselines['Buy_Hold'] = np.array(rets)

    # Market-Cap Weighted
    mkt_col = 'mkt_cap' if 'mkt_cap' in df.columns else ('MKT_CAP' if 'MKT_CAP' in df.columns else None)
    if mkt_col:
        rets = []
        for t in range(1, len(dates)):
            day_data = df[df['date'] == dates[t]].sort_values('PERMNO')
            if len(day_data) == n_assets:
                mc = np.nan_to_num(day_data[mkt_col].values.astype(float), nan=0.0)
                mc = np.maximum(mc, 0)
                w = mc / mc.sum() if mc.sum() > 0 else np.ones(n_assets) / n_assets
            else:
                w = np.ones(n_assets) / n_assets
            rets.append(np.dot(w, ret_matrix[t]))
        baselines['MktCap_Weight'] = np.array(rets)

    return {name: (compute_metrics(rets), rets) for name, rets in baselines.items()}


# Load data


In [ ]:
parts = []
for fname in [TRAIN_DATA_PATH, VAL_DATA_PATH, TEST_DATA_PATH]:
    df = pd.read_csv(fname, parse_dates=['date'])
    parts.append(df)
    print(f"  {fname}: {len(df)} rows, "
          f"{df['date'].min().date()} to {df['date'].max().date()}")
full_df = pd.concat(parts, ignore_index=True)
full_df = full_df.sort_values(['date', 'PERMNO']).drop_duplicates(['date', 'PERMNO']).reset_index(drop=True)
print(f"  Combined: {len(full_df)} rows, "
      f"{full_df['date'].min().date()} to {full_df['date'].max().date()}")

# ---- Determine the 47-stock universe ----
train_only = parts[0]
val_only = parts[1]
train_dates = sorted(train_only['date'].unique())
train_cov = train_only.groupby('PERMNO')['date'].nunique()
full_train_stocks = set(train_cov[train_cov == len(train_dates)].index)
val_dates = sorted(val_only['date'].unique())
val_cov = val_only.groupby('PERMNO')['date'].nunique()
full_val_stocks = set(val_cov[val_cov == len(val_dates)].index)
universe_47 = sorted(full_train_stocks & full_val_stocks)
print(f"  Universe (47 stocks): {len(universe_47)} PERMNOs")
assert len(universe_47) == 47, f"Expected 47 stocks, got {len(universe_47)}"

full_df = full_df[full_df['PERMNO'].isin(universe_47)].copy()

# Reconstruct raw_RET using the scaler fit once on Pipeline A train
if 'raw_RET' not in full_df.columns:
    if os.path.exists(SCALER_PARAMS_PATH):
        scaler = pd.read_csv(SCALER_PARAMS_PATH, index_col=0)
        ret_mean = scaler.loc['RET', 'mean']
        ret_std = scaler.loc['RET', 'std']
        full_df['raw_RET'] = full_df['RET'] * ret_std + ret_mean
        print(f"  Denormalized RET using scaler_params.csv (mean={ret_mean:.6f}, std={ret_std:.6f})")
    else:
        print("  WARNING: raw_RET missing and no scaler_params.csv found")


# Walk-forward loop


In [ ]:
all_trials = []          # per-window PPO trial DataFrames (15 rows each)
all_test_rows = []       # PPO test row, one per window
test_daily_returns = {}  # window_id -> {algo -> daily_returns}

for window in WINDOWS:
    wid = window['id']
    print("\n" + "=" * 78)
    print(f"  WINDOW {wid}/6")
    print(f"    Train: {window['train_start']} → {window['train_end']}")
    print(f"    Val:   {window['val_start']}   → {window['val_end']}")
    print(f"    Test:  {window['test_start']}  → {window['test_end']}")
    print("=" * 78)

    test_df = full_df[(full_df['date'] >= window['test_start']) &
                      (full_df['date'] <= window['test_end'])].copy()
    print(f"  Test: {len(test_df)} rows, {test_df['date'].nunique()} days, "
          f"{test_df['PERMNO'].nunique()} stocks")

    # ---- Load per-window PPO trial log and pick best by val Sharpe ----
    trials_path = f'results/pipeline_b_w{wid}_ppo_trials.csv'
    trials_df = pd.read_csv(trials_path)
    all_trials.append(trials_df)
    best_row = trials_df.loc[trials_df['sharpe'].astype(float).idxmax()]
    best_hp = {'timesteps': int(best_row['timesteps']),
               'learning_rate': float(best_row['learning_rate'])}
    best_seed = int(best_row['seed'])
    best_val_sharpe = float(best_row['sharpe'])
    print(f"  PPO BEST (window {wid}): trial #{int(best_row['trial'])}, "
          f"ts={best_hp['timesteps']}, lr={best_hp['learning_rate']:.2e}, "
          f"seed={best_seed}, val_Sharpe={best_val_sharpe:.4f}")

    # ---- Baselines on test period ----
    print(f"\n  --- Baselines on test year {window['test_start'][:4]} ---")
    baseline_results = run_baselines_on_period(test_df)
    for name, (metrics, rets) in baseline_results.items():
        print(f"    {name}: Cum={metrics['cumulative_return']}%, "
              f"Sharpe={metrics['sharpe']}, MaxDD={metrics['max_drawdown']}%")
        test_daily_returns.setdefault(wid, {})[name] = rets

    # ---- Load saved final PPO agent and evaluate on test ----
    model_path = os.path.join(MODEL_DIR, f"PPO_window{wid}_final")
    if not (os.path.exists(model_path) or os.path.exists(model_path + '.zip')):
        raise FileNotFoundError(
            f"PPO model not found: {model_path}.zip — Pipeline B requires the six "
            f"PPO_window{{1..6}}_final.zip files in {MODEL_DIR}/. Train Pipeline B or copy them in."
        )
    model = PPO.load(model_path)
    ppo_test_metrics, ppo_test_rets = evaluate_model(model, test_df)
    print(f"  PPO test (window {wid}): Sharpe={ppo_test_metrics['sharpe']:.4f}, "
          f"Cum={ppo_test_metrics['cumulative_return']}%, "
          f"MaxDD={ppo_test_metrics['max_drawdown']}%")
    all_test_rows.append({
        'window': wid, 'algorithm': 'PPO',
        'timesteps': best_hp['timesteps'],
        'learning_rate': best_hp['learning_rate'],
        'seed': best_seed,
        'val_sharpe': best_val_sharpe,
        **ppo_test_metrics,
    })
    test_daily_returns.setdefault(wid, {})['PPO'] = ppo_test_rets
    del model

ppo_trials_df = pd.concat(all_trials, ignore_index=True)
ppo_test_df = pd.DataFrame(all_test_rows)
print(f"\n  PPO trials accumulated: {len(ppo_trials_df)} rows ({ppo_trials_df['window'].nunique()} windows × {N_TRIALS} trials)")
print(f"  PPO test rows: {len(ppo_test_df)} (one per window)")


# Read-merge shared outputs


In [ ]:
# Per-window PPO trial CSVs (PPO-only by name, no merge needed)
for wid in sorted(ppo_trials_df['window'].unique()):
    win_trials = ppo_trials_df[ppo_trials_df['window'] == wid]
    out_path = f'results/pipeline_b_w{wid}_ppo_trials.csv'
    win_trials.to_csv(out_path, index=False)
    print(f"  Wrote {out_path} ({len(win_trials)} rows)")

# Combined pipeline_b_trials.csv: read-merge — keep non-PPO rows, write PPO rows
if os.path.exists(TRIALS_CSV):
    existing = pd.read_csv(TRIALS_CSV)
    non_ppo = existing[existing['algorithm'] != 'PPO']
else:
    non_ppo = pd.DataFrame(columns=ppo_trials_df.columns)
combined_trials = pd.concat([non_ppo, ppo_trials_df], ignore_index=True)
combined_trials = combined_trials.sort_values(['window', 'algorithm', 'trial']).reset_index(drop=True)
combined_trials.to_csv(TRIALS_CSV, index=False)
print(f"  Wrote {TRIALS_CSV} ({len(combined_trials)} rows: {len(non_ppo)} non-PPO + {len(ppo_trials_df)} PPO)")

# pipeline_b_test_results.csv: read-merge — keep non-PPO rows (baselines + DQN), write PPO rows
if os.path.exists(TEST_RESULTS_CSV):
    existing_test = pd.read_csv(TEST_RESULTS_CSV)
    non_ppo_test = existing_test[existing_test['algorithm'] != 'PPO']
else:
    non_ppo_test = pd.DataFrame()
combined_test = pd.concat([non_ppo_test, ppo_test_df], ignore_index=True)
combined_test = combined_test.sort_values(['window', 'algorithm']).reset_index(drop=True)
combined_test.to_csv(TEST_RESULTS_CSV, index=False)
print(f"  Wrote {TEST_RESULTS_CSV} ({len(combined_test)} rows: {len(non_ppo_test)} non-PPO + {len(ppo_test_df)} PPO)")


# Aggregate summary


In [ ]:
results_df = pd.read_csv(TEST_RESULTS_CSV)

summary_rows = []
for algo in ['Equal_Weight', 'Buy_Hold', 'MktCap_Weight', 'PPO', 'DQN']:
    algo_rows = results_df[results_df['algorithm'] == algo]
    if len(algo_rows) == 0:
        continue
    sharpes = algo_rows['sharpe'].astype(float).values
    cum_returns = algo_rows['cumulative_return'].astype(float).values
    max_dds = algo_rows['max_drawdown'].astype(float).values

    summary_rows.append({
        'algorithm': algo,
        'n_windows': len(sharpes),
        'mean_sharpe': round(np.mean(sharpes), 4),
        'std_sharpe': round(np.std(sharpes, ddof=1), 4) if len(sharpes) > 1 else 0,
        'min_sharpe': round(np.min(sharpes), 4),
        'max_sharpe': round(np.max(sharpes), 4),
        'mean_cum_return': round(np.mean(cum_returns), 2),
        'mean_max_dd': round(np.mean(max_dds), 2),
    })

summary_df = pd.DataFrame(summary_rows)
print("\n  Per-algorithm Sharpe across 6 windows:")
print(summary_df.to_string(index=False))

summary_df.to_csv(SUMMARY_CSV, index=False)
print(f"\n  Wrote {SUMMARY_CSV}")


# T-tests vs Equal_Weight


In [ ]:
ew_sharpes = results_df[results_df['algorithm'] == 'Equal_Weight']\
    .sort_values('window')['sharpe'].astype(float).values

ttest_rows = []
print("\n  T-tests (paired, one-sided: algo > Equal_Weight):")
for algo in ['PPO', 'MktCap_Weight']:
    algo_sharpes = results_df[results_df['algorithm'] == algo]\
        .sort_values('window')['sharpe'].astype(float).values
    if len(algo_sharpes) == len(ew_sharpes) and len(algo_sharpes) > 1:
        t_stat, p_val_two_sided = stats.ttest_rel(algo_sharpes, ew_sharpes)
        p_val_one_sided = p_val_two_sided / 2 if t_stat > 0 else 1 - p_val_two_sided / 2
        print(f"    {algo:<15} vs Equal_Weight: "
              f"t={t_stat:.4f}, p_one-sided={p_val_one_sided:.4f}")
        ttest_rows.append({
            'algorithm': algo,
            'mean_diff': round(float(np.mean(algo_sharpes - ew_sharpes)), 4),
            't_stat': round(float(t_stat), 4),
            'p_two_sided': round(float(p_val_two_sided), 4),
            'p_one_sided': round(float(p_val_one_sided), 4),
        })

ttest_df = pd.DataFrame(ttest_rows)

# pipeline_b_summary.xlsx: read-merge — preserve DQN ttests row (from NB6), drop the rest, add PPO + MktCap
ttests_existing = pd.DataFrame()
if os.path.exists(SUMMARY_XLSX):
    try:
        ttests_existing = pd.read_excel(SUMMARY_XLSX, sheet_name='ttests')
    except Exception:
        ttests_existing = pd.DataFrame()
if 'algorithm' in ttests_existing.columns:
    dqn_rows = ttests_existing[ttests_existing['algorithm'] == 'DQN']
else:
    dqn_rows = pd.DataFrame()
ttests_merged = pd.concat([ttest_df, dqn_rows], ignore_index=True)

with pd.ExcelWriter(SUMMARY_XLSX, engine='openpyxl') as xw:
    summary_df.to_excel(xw, sheet_name='per_algo_summary', index=False)
    ttests_merged.to_excel(xw, sheet_name='ttests', index=False)
    results_df.to_excel(xw, sheet_name='per_window_results', index=False)
print(f"  Wrote {SUMMARY_XLSX}")
